# Master's Thesis Analysis: LLM Stock Price Prediction

This notebook loads the raw data collected from the `stock_predictions_log.jsonl` file, extracts the relevant prediction, confidence, and LLM usage metrics, and generates visualizations and descriptive statistics for thesis inclusion.


In [1]:
import pandas as pd
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# Set file path (MANDATORY: Ensure this file exists)
LOG_FILE = 'stock_predictions_log.jsonl'

#LOG_FILE = 'test.jsonl'

# Set plotting style for academic presentation
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

print(f"Setup complete. Data will be loaded from: {LOG_FILE}")


Setup complete. Data will be loaded from: stock_predictions_log.jsonl


## 1. Data Ingestion and Robust Parsing

The parsing function has been updated to handle the nested structure of the LLM response object more reliably.

In [2]:
def extract_data_from_entry(entry):
    """Parses a single JSON line entry and extracts structured prediction data."""
    
    # Base data extraction
    data = {
        'collection_timestamp': entry.get('collection_timestamp'),
        'ticker': entry.get('ticker'),
        'start_price': entry.get('start_price')
    }
    
    # Initialize prediction fields to NaN/default in case of parsing failure
    data.update({
        'pred_day': np.nan, 'pred_week': np.nan, 'pred_month': np.nan,
        'confidence': 'Unknown', 'reasoning': '',
        'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'num_sources': 0
    })
    
    llm_dump = entry.get('llm_response_dump', {})

    # --- Robustly Extract LLM Output Text ---
    raw_text = ''
    try:
        # Look for message type output which contains the actual text response
        output_list = llm_dump.get('output', [])
        
        for output_item in output_list:
            if output_item.get('type') == 'message':
                # Found the message, now get the content
                content_list = output_item.get('content', [])
                for content_item in content_list:
                    if content_item.get('type') == 'output_text':
                        raw_text = content_item.get('text', '')
                        break
                if raw_text:
                    break
        
        # The actual format in the data has specific patterns
        # Looking for patterns like: "PREDICTED_PRICE_DAY: 120.30"
        day_match = re.search(r"PREDICTED_PRICE_DAY:\s*([\d.]+)", raw_text, re.IGNORECASE)
        week_match = re.search(r"PREDICTED_PRICE_WEEK:\s*([\d.]+)", raw_text, re.IGNORECASE)
        month_match = re.search(r"PREDICTED_PRICE_MONTH:\s*([\d.]+)", raw_text, re.IGNORECASE)
        confidence_match = re.search(r"CONFIDENCE:\s*(High|Medium|Low)", raw_text, re.IGNORECASE)
        reasoning_match = re.search(r"REASONING:\s*(.+?)(?=\n|$)", raw_text, re.IGNORECASE | re.DOTALL)

        # Update data with parsed values
        data['pred_day'] = float(day_match.group(1)) if day_match else data['pred_day']
        data['pred_week'] = float(week_match.group(1)) if week_match else data['pred_week']
        data['pred_month'] = float(month_match.group(1)) if month_match else data['pred_month']
        data['confidence'] = confidence_match.group(1).capitalize() if confidence_match else data['confidence']
        data['reasoning'] = reasoning_match.group(1).strip() if reasoning_match else data['reasoning']
        
    except Exception as e:
        print(f"⚠️ Error parsing text output for {data.get('ticker')} on day {data.get('simulated_day')}: {e}")
        
    # --- Extract LLM Usage and Grounding Metadata ---
    try:
        usage = llm_dump.get('usage', {})
        data['prompt_tokens'] = usage.get('input_tokens', 0)
        data['completion_tokens'] = usage.get('output_tokens', 0)
        data['total_tokens'] = usage.get('total_tokens', 0)
        
        # Count the number of search sources used for grounding
        # Sources are in web_search_call type outputs
        total_sources = 0
        for output_item in llm_dump.get('output', []):
            if output_item.get('type') == 'web_search_call':
                action = output_item.get('action', {})
                sources = action.get('sources', [])
                total_sources += len(sources) if sources is not None else 0 #some search calls have 'null' as scurces and query, which would cause an error, resulting in 0 for total_sources
        data['num_sources'] = total_sources
        
    except Exception:
        pass # Keep default values (0)
        
    return data


try:
    all_entries = []
    with open(LOG_FILE, 'r') as f:
        for line in f:
            all_entries.append(json.loads(line))
    
    # Convert raw entries into the structured DataFrame
    df = pd.DataFrame([extract_data_from_entry(e) for e in all_entries])
    
    # Calculate relative change for each horizon
    df['pred_change_day_%'] = ((df['pred_day'] - df['start_price']) / df['start_price']) * 100
    df['pred_change_week_%'] = ((df['pred_week'] - df['start_price']) / df['start_price']) * 100
    df['pred_change_month_%'] = ((df['pred_month'] - df['start_price']) / df['start_price']) * 100
    
    print(f"\n✅ Successfully loaded and processed {len(df)} records.")
    print("\nData Preview:")
    print(df.head())
    
    # Print summary of parsing success
    print(f"\n📊 Parsing Summary:")
    print(f"   - Records with valid day predictions: {df['pred_day'].notna().sum()}")
    print(f"   - Records with valid week predictions: {df['pred_week'].notna().sum()}")
    print(f"   - Records with valid month predictions: {df['pred_month'].notna().sum()}")
    
except FileNotFoundError:
    print(f"❌ ERROR: The data file '{LOG_FILE}' was not found. Please ensure the collection script was run successfully.")
    df = pd.DataFrame()
except Exception as e:
    print(f"❌ CRITICAL ERROR processing data: {e}")
    df = pd.DataFrame()


✅ Successfully loaded and processed 899 records.

Data Preview:
         collection_timestamp ticker  start_price  pred_day  pred_week  \
0  2025-11-10T15:56:30.213013    LEN       121.50     121.9      123.6   
1  2025-11-10T15:57:08.074150    ADM        56.17      56.4       56.6   
2  2025-11-10T15:57:59.927649   NDAQ        87.23      87.4       88.4   
3  2025-11-10T15:58:52.227615   SNPS       390.95     395.5      405.5   
4  2025-11-10T15:59:42.731194   IDXX       701.25     704.5      722.0   

   pred_month confidence                                          reasoning  \
0       126.5     Medium  Near-term Lennar moves are driven by a stabili...   
1        57.9     Medium  Q3 2025 results and updated guidance triggered...   
2        90.6     Medium  Near-term price action is driven by macro poli...   
3       420.5     Medium  Bullish market momentum and AI-driven demand f...   
4       735.0     Medium  Positive Q3 results and raised full-year guida...   

   prompt_token

In [3]:
def get_close_price(ticker, target_date):
    """Fetch closing price for a ticker on or before target_date."""
    # Fetch a window around the target date to handle weekends/holidays
    start = target_date - timedelta(days=5)
    end = target_date + timedelta(days=1)
    
    data = yf.download(ticker, start=start.strftime('%Y-%m-%d'), end=end.strftime('%Y-%m-%d'), 
                       progress=False, auto_adjust=True)
    if data.empty:
        return None
    
    # Use previous available trading day if exact date not found
    available = data[data.index <= pd.Timestamp(target_date)]
    if available.empty:
        return None
    
    return float(available['Close'].iloc[-1].iloc[0])


# Parse timestamps once
df['collection_timestamp'] = pd.to_datetime(df['collection_timestamp'])

# Columns to add
new_cols = {
    'actual_close_day':     [],
    'actual_change_day_%':  [],
    'actual_close_week':    [],
    'actual_change_week_%': [],
    'actual_close_month':   [],
    'actual_change_month_%':[],
}

for _, row in df.iterrows():
    base_date  = row['collection_timestamp'].date()
    start_price = row['start_price']
    ticker      = row['ticker']

    date_day   = base_date + timedelta(days=1)
    date_week  = base_date + timedelta(days=7)
    date_month = base_date + relativedelta(months=1)

    close_day   = get_close_price(ticker, date_day)
    close_week  = get_close_price(ticker, date_week)
    close_month = get_close_price(ticker, date_month)

    new_cols['actual_close_day'].append(close_day)
    new_cols['actual_change_day_%'].append(
        round((close_day - start_price) / start_price * 100, 6) if close_day else None)

    new_cols['actual_close_week'].append(close_week)
    new_cols['actual_change_week_%'].append(
        round((close_week - start_price) / start_price * 100, 6) if close_week else None)

    new_cols['actual_close_month'].append(close_month)
    new_cols['actual_change_month_%'].append(
        round((close_month - start_price) / start_price * 100, 6) if close_month else None)

# Assign all new columns at once
for col, values in new_cols.items():
    df[col] = values

print("Done! New columns added:")
print(df[['ticker', 'start_price', 'actual_close_day', 'actual_change_day_%',
          'actual_close_week', 'actual_change_week_%',
          'actual_close_month', 'actual_change_month_%']].head())

Done! New columns added:
  ticker  start_price  actual_close_day  actual_change_day_%  \
0    LEN       121.50        124.882637             2.784063   
1    ADM        56.17         56.720890             0.980755   
2   NDAQ        87.23         87.290932             0.069852   
3   SNPS       390.95        395.600006             1.189412   
4   IDXX       701.25        712.469971             1.599996   

   actual_close_week  actual_change_week_%  actual_close_month  \
0         113.593018             -6.507804          119.496674   
1          57.813381              2.925728           57.774536   
2          85.173996             -2.356992           91.629829   
3         390.239990             -0.181611          475.829987   
4         669.000000             -4.598930          701.830017   

   actual_change_month_%  
0              -1.648828  
1               2.856571  
2               5.043941  
3              21.711213  
4               0.082712  


In [4]:
df.info()
print(df.head().to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899 entries, 0 to 898
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   collection_timestamp   899 non-null    datetime64[ns]
 1   ticker                 899 non-null    object        
 2   start_price            899 non-null    float64       
 3   pred_day               899 non-null    float64       
 4   pred_week              899 non-null    float64       
 5   pred_month             899 non-null    float64       
 6   confidence             899 non-null    object        
 7   reasoning              899 non-null    object        
 8   prompt_tokens          899 non-null    int64         
 9   completion_tokens      899 non-null    int64         
 10  total_tokens           899 non-null    int64         
 11  num_sources            899 non-null    int64         
 12  change_day_%           899 non-null    float64       
 13  chang

In [7]:
#df.to_csv('my_data.csv', index=False)

In [6]:
# Filter the dataframe for November 28
# (Month 11 = November, Day 28 Thanksgiving)
nov_28_reasoning = df[(df['collection_timestamp'].dt.month == 11) & 
                      (df['collection_timestamp'].dt.day == 28)]['reasoning']

# Print each reasoning entry
for entry in nov_28_reasoning:
    print(entry)
    print("-" * 30) # Separator for readability

Near-term Lennar price is driven by anticipated mortgage-rate relief and ongoing housing-market momentum, with the broader macro backdrop supporting modest upside across the three horizons. ([nasdaq.com](https://www.nasdaq.com/press-release/mortgage-rates-decrease-2025-11-26?utm_source=openai))
------------------------------
ADM's Q3 2025 results and revised full-year guidance toward lower earnings, due to weaker crush margins and ongoing biofuel/trade policy uncertainty, are the primary driver of all three horizons.
------------------------------
Momentum from Nasdaq's solid Q3 2025 results in a constructive market backdrop, with holiday liquidity likely capping large moves.
------------------------------
The three horizons are primarily driven by near-term momentum from the stock's movement and the upside implied by analysts' price targets, with Synopsys' upcoming earnings acting as a key catalyst. ([marketwatch.com](https://www.marketwatch.com/data-news/synopsys-inc-stock-outperform

# FICO talk about weekend, FCX, AZO, JBL have December 1. as next trading day.